### Workflow
1. Replace 2026.csv with the updated latest file.
2. Re-run your feature engineering notebook from the top.
3. Export the processed match_df and neutral_df.
4. Open the bracket producer file.
5. Load processed match_df, neutral_df, and the Wimbledon draw CSV.
6. Run the bracket producer.
7. It outputs 3 champion-constrained optimized brackets.


#### Expected draw format

CSV file with 

slot,player_name,player_id,seed

only slot and player_name required

In [21]:
import pandas as pd
import json

INPUT_DIR = "bracket_producer_inputs"

match_df = pd.read_pickle(f"{INPUT_DIR}/processed_match_df.pkl")
neutral_df = pd.read_pickle(f"{INPUT_DIR}/processed_neutral_df.pkl")

with open(f"{INPUT_DIR}/model_feature_cols.json", "r") as f:
    MODEL_FEATURE_COLS = json.load(f)

draw_df = pd.read_csv(f"{INPUT_DIR}/wimbledon_2026_draw.csv")

# Manual draw name / ID fixes

if "player_id" not in draw_df.columns:
    draw_df["player_id"] = pd.NA

draw_df.loc[
    draw_df["player_name"] == "Aleksandr Shevchenko",
    "player_name"
] = "Alexander Shevchenko"

draw_df.loc[
    draw_df["player_name"] == "Alexander Shevchenko",
    "player_id"
] = "S0H2"

## General Tournament Config

In [5]:
GRAND_SLAM_CONFIG = {
    "Australian Open": {
        "surface": "Hard",
        "draw_size": 128
    },
    "Roland Garros": {
        "surface": "Clay",
        "draw_size": 128
    },
    "Wimbledon": {
        "surface": "Grass",
        "draw_size": 128
    },
    "US Open": {
        "surface": "Hard",
        "draw_size": 128
    }
}

ROUND_POINTS = {
    "R128": 10,
    "R64": 20,
    "R32": 40,
    "R16": 80,
    "QF": 160,
    "SF": 320,
    "F": 640
}

ROUND_SEQUENCE_128 = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]

MODEL_FEATURE_COLS = [
    "elo_diff",
    "surface_elo_diff",
    "score_elo_diff",
    "score_surface_elo_diff",
    "rank_diff",
    "log_rank_points_diff",
    "last10_win_pct_pre_diff",
    "last20_win_pct_pre_diff",
    "surface_last10_win_pct_pre_diff",
    "surface_last20_win_pct_pre_diff",
    "last10_matches_pre_diff",
    "last20_matches_pre_diff",
    "surface_last10_matches_pre_diff",
    "surface_last20_matches_pre_diff"
]

### Traning Model before tournament start date

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

def train_model_before_date(neutral_df, cutoff_date, feature_cols):
    """
    Trains the match prediction model using only matches before cutoff_date.
    """
    
    cutoff_date = pd.to_datetime(cutoff_date)
    
    train = neutral_df[
        neutral_df["date"] < cutoff_date
    ].dropna(subset=["A_won"]).copy()
    
    X_train = train[feature_cols]
    y_train = train["A_won"]
    
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=5000))
    ])
    
    model.fit(X_train, y_train)
    
    print("Training cutoff:", cutoff_date.date())
    print("Training matches:", len(train))
    
    return model

### Current player features before tournament

In [7]:
import numpy as np
import pandas as pd
import unicodedata


def normalize_player_name(name):
    """
    Normalizes player names for safer matching.
    """
    
    if pd.isna(name):
        return ""
    
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join([c for c in name if not unicodedata.combining(c)])
    name = " ".join(name.split())
    
    return name


def expected_win_prob(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def get_recent_win_pct(history, window):
    recent = history[-window:]
    if len(recent) == 0:
        return 0.5
    return np.mean(recent)


def build_current_player_features_before_date(
    match_df,
    cutoff_date,
    target_surface,
    start_elo=1500,
    k=32
):
    """
    Builds current player features as of cutoff_date.
    
    This recomputes current Elo, surface Elo, score-adjusted Elo,
    recent form, and latest known rank/rank points using only prior matches.
    """
    
    cutoff_date = pd.to_datetime(cutoff_date)
    
    df = match_df[
        match_df["tourney_date"] < cutoff_date
    ].copy()
    
    df = df.sort_values(
        ["tourney_date", "tourney_id", "match_num"]
    ).reset_index(drop=True)
    
    overall_ratings = {}
    surface_ratings = {}
    score_ratings = {}
    score_surface_ratings = {}
    
    player_history = {}
    player_surface_history = {}
    
    latest_meta = {}
    
    for _, row in df.iterrows():
        winner = row["winner_id"]
        loser = row["loser_id"]
        surface = row["surface"]
        
        if pd.isna(surface):
            surface = "Unknown"
        
        margin_mult = row.get("score_margin_mult", 1.0)
        if pd.isna(margin_mult):
            margin_mult = 1.0
        
        # ----------------------------
        # Latest metadata
        # ----------------------------
        
        latest_meta[winner] = {
            "player_id": winner,
            "name": row["winner_name"],
            "rank": row["winner_rank"],
            "rank_points": row["winner_rank_points"],
            "age": row.get("winner_age", np.nan),
            "hand": row.get("winner_hand", np.nan),
            "ht": row.get("winner_ht", np.nan)
        }
        
        latest_meta[loser] = {
            "player_id": loser,
            "name": row["loser_name"],
            "rank": row["loser_rank"],
            "rank_points": row["loser_rank_points"],
            "age": row.get("loser_age", np.nan),
            "hand": row.get("loser_hand", np.nan),
            "ht": row.get("loser_ht", np.nan)
        }
        
        # ----------------------------
        # Overall Elo
        # ----------------------------
        
        w_rating = overall_ratings.get(winner, start_elo)
        l_rating = overall_ratings.get(loser, start_elo)
        
        p_w = expected_win_prob(w_rating, l_rating)
        change = k * (1 - p_w)
        
        overall_ratings[winner] = w_rating + change
        overall_ratings[loser] = l_rating - change
        
        # ----------------------------
        # Surface Elo
        # ----------------------------
        
        w_key = (winner, surface)
        l_key = (loser, surface)
        
        w_surface_rating = surface_ratings.get(w_key, start_elo)
        l_surface_rating = surface_ratings.get(l_key, start_elo)
        
        p_w_surface = expected_win_prob(w_surface_rating, l_surface_rating)
        surface_change = k * (1 - p_w_surface)
        
        surface_ratings[w_key] = w_surface_rating + surface_change
        surface_ratings[l_key] = l_surface_rating - surface_change
        
        # ----------------------------
        # Score-adjusted Elo
        # ----------------------------
        
        w_score_rating = score_ratings.get(winner, start_elo)
        l_score_rating = score_ratings.get(loser, start_elo)
        
        p_w_score = expected_win_prob(w_score_rating, l_score_rating)
        score_change = k * margin_mult * (1 - p_w_score)
        
        score_ratings[winner] = w_score_rating + score_change
        score_ratings[loser] = l_score_rating - score_change
        
        # ----------------------------
        # Score-adjusted surface Elo
        # ----------------------------
        
        w_score_surface_rating = score_surface_ratings.get(w_key, start_elo)
        l_score_surface_rating = score_surface_ratings.get(l_key, start_elo)
        
        p_w_score_surface = expected_win_prob(
            w_score_surface_rating,
            l_score_surface_rating
        )
        
        score_surface_change = k * margin_mult * (1 - p_w_score_surface)
        
        score_surface_ratings[w_key] = (
            w_score_surface_rating + score_surface_change
        )
        
        score_surface_ratings[l_key] = (
            l_score_surface_rating - score_surface_change
        )
        
        # ----------------------------
        # Recent form histories
        # ----------------------------
        
        player_history.setdefault(winner, []).append(1)
        player_history.setdefault(loser, []).append(0)
        
        player_surface_history.setdefault((winner, surface), []).append(1)
        player_surface_history.setdefault((loser, surface), []).append(0)
    
    # ----------------------------
    # Build final player feature dictionary
    # ----------------------------
    
    player_features = {}
    
    for pid, meta in latest_meta.items():
        overall_hist = player_history.get(pid, [])
        surface_hist = player_surface_history.get((pid, target_surface), [])
        
        player_features[pid] = {
            "player_id": pid,
            "name": meta["name"],
            "rank": meta["rank"],
            "rank_points": meta["rank_points"],
            "age": meta["age"],
            "hand": meta["hand"],
            "ht": meta["ht"],
            "elo": overall_ratings.get(pid, start_elo),
            "surface_elo": surface_ratings.get((pid, target_surface), start_elo),
            "score_elo": score_ratings.get(pid, start_elo),
            "score_surface_elo": score_surface_ratings.get((pid, target_surface), start_elo),
            "last10_win_pct": get_recent_win_pct(overall_hist, 10),
            "last20_win_pct": get_recent_win_pct(overall_hist, 20),
            "last10_matches": min(len(overall_hist), 10),
            "last20_matches": min(len(overall_hist), 20),
            "surface_last10_win_pct": get_recent_win_pct(surface_hist, 10),
            "surface_last20_win_pct": get_recent_win_pct(surface_hist, 20),
            "surface_last10_matches": min(len(surface_hist), 10),
            "surface_last20_matches": min(len(surface_hist), 20)
        }
    
    return player_features

### Match draw names to player features

In [8]:
def make_unknown_player(slot, player_name, target_surface):
    """
    Creates a default player profile for qualifiers/unmatched players.
    """
    
    unknown_id = f"UNKNOWN_SLOT_{slot}"
    
    return {
        "player_id": unknown_id,
        "name": player_name,
        "rank": np.nan,
        "rank_points": np.nan,
        "age": np.nan,
        "hand": np.nan,
        "ht": np.nan,
        "elo": 1500,
        "surface_elo": 1500,
        "score_elo": 1500,
        "score_surface_elo": 1500,
        "last10_win_pct": 0.5,
        "last20_win_pct": 0.5,
        "last10_matches": 0,
        "last20_matches": 0,
        "surface_last10_win_pct": 0.5,
        "surface_last20_win_pct": 0.5,
        "surface_last10_matches": 0,
        "surface_last20_matches": 0
    }


def match_draw_to_player_features(draw_df, current_player_features, target_surface):
    """
    Matches draw slots to current player features.
    
    Uses player_id if available, otherwise normalized player_name.
    """
    
    draw_df = draw_df.copy()
    draw_df = draw_df.sort_values("slot").reset_index(drop=True)
    
    # Name lookup
    name_lookup = {}
    
    for pid, info in current_player_features.items():
        norm_name = normalize_player_name(info["name"])
        name_lookup[norm_name] = pid
    
    draw_players = {}
    match_rows = []
    
    for _, row in draw_df.iterrows():
        slot = int(row["slot"])
        player_name = row["player_name"]
        
        player_id = row.get("player_id", np.nan)
        
        matched_id = None
        match_method = None
        
        if "player_id" in draw_df.columns and not pd.isna(player_id):
            try:
                player_id_int = int(player_id)
            except Exception:
                player_id_int = player_id
            
            if player_id_int in current_player_features:
                matched_id = player_id_int
                match_method = "player_id"
        
        if matched_id is None:
            norm_name = normalize_player_name(player_name)
            
            if norm_name in name_lookup:
                matched_id = name_lookup[norm_name]
                match_method = "name"
        
        if matched_id is None:
            player = make_unknown_player(slot, player_name, target_surface)
            matched_id = player["player_id"]
            match_method = "unknown_default"
        else:
            player = current_player_features[matched_id].copy()
        
        # Make slot-specific copy
        player["draw_slot"] = slot
        player["display_name"] = player_name
        
        draw_players[matched_id] = player
        
        match_rows.append({
            "slot": slot,
            "draw_player_name": player_name,
            "matched_player_id": matched_id,
            "matched_player_name": player["name"],
            "match_method": match_method,
            "rank": player["rank"],
            "rank_points": player["rank_points"],
            "surface_elo": player["surface_elo"],
            "score_surface_elo": player["score_surface_elo"]
        })
    
    match_report = pd.DataFrame(match_rows)
    
    return draw_players, match_report

### Build bracket tree from draw slots

In [9]:
def build_draw_bracket_tree(draw_players, draw_size=128):
    """
    Builds a bracket tree from draw slots.
    
    Assumes standard Grand Slam bracket order:
    slots 1 vs 2, 3 vs 4, ..., 127 vs 128.
    """
    
    if draw_size != 128:
        raise ValueError("This version currently supports 128-player Grand Slam draws.")
    
    slot_to_pid = {
        info["draw_slot"]: pid
        for pid, info in draw_players.items()
    }
    
    missing_slots = sorted(set(range(1, draw_size + 1)) - set(slot_to_pid.keys()))
    
    if missing_slots:
        raise ValueError(f"Missing draw slots: {missing_slots}")
    
    nodes_by_round = {}
    
    # First round
    r128_nodes = []
    
    for i in range(1, draw_size + 1, 2):
        p1_id = slot_to_pid[i]
        p2_id = slot_to_pid[i + 1]
        
        node = {
            "round": "R128",
            "round_index": len(r128_nodes),
            "p1_id": p1_id,
            "p2_id": p2_id,
            "left": None,
            "right": None
        }
        
        r128_nodes.append(node)
    
    nodes_by_round["R128"] = r128_nodes
    
    # Higher rounds
    previous_nodes = r128_nodes
    
    for rnd in ROUND_SEQUENCE_128[1:]:
        current_nodes = []
        
        for i in range(0, len(previous_nodes), 2):
            node = {
                "round": rnd,
                "round_index": len(current_nodes),
                "p1_id": None,
                "p2_id": None,
                "left": previous_nodes[i],
                "right": previous_nodes[i + 1]
            }
            
            current_nodes.append(node)
        
        nodes_by_round[rnd] = current_nodes
        previous_nodes = current_nodes
    
    root = nodes_by_round["F"][0]
    
    return root, nodes_by_round

### Prediction features and probability funcion

In [10]:
def make_matchup_features(player_A, player_B, feature_cols):
    """
    Creates one row of model features for Player A vs Player B.
    """
    
    row = {
        "elo_diff": player_A["elo"] - player_B["elo"],
        "surface_elo_diff": player_A["surface_elo"] - player_B["surface_elo"],
        "score_elo_diff": player_A["score_elo"] - player_B["score_elo"],
        "score_surface_elo_diff": player_A["score_surface_elo"] - player_B["score_surface_elo"],
        
        "rank_diff": player_B["rank"] - player_A["rank"],
        
        "log_rank_points_diff": (
            np.log1p(player_A["rank_points"]) 
            - np.log1p(player_B["rank_points"])
        ),
        
        "last10_win_pct_pre_diff": (
            player_A["last10_win_pct"] - player_B["last10_win_pct"]
        ),
        
        "last20_win_pct_pre_diff": (
            player_A["last20_win_pct"] - player_B["last20_win_pct"]
        ),
        
        "surface_last10_win_pct_pre_diff": (
            player_A["surface_last10_win_pct"] - player_B["surface_last10_win_pct"]
        ),
        
        "surface_last20_win_pct_pre_diff": (
            player_A["surface_last20_win_pct"] - player_B["surface_last20_win_pct"]
        ),
        
        "last10_matches_pre_diff": (
            player_A["last10_matches"] - player_B["last10_matches"]
        ),
        
        "last20_matches_pre_diff": (
            player_A["last20_matches"] - player_B["last20_matches"]
        ),
        
        "surface_last10_matches_pre_diff": (
            player_A["surface_last10_matches"] - player_B["surface_last10_matches"]
        ),
        
        "surface_last20_matches_pre_diff": (
            player_A["surface_last20_matches"] - player_B["surface_last20_matches"]
        )
    }
    
    return pd.DataFrame([row])[feature_cols]


def predict_player_A_win_prob(model, player_A, player_B, feature_cols):
    """
    Predicts P(Player A beats Player B).
    """
    
    X = make_matchup_features(player_A, player_B, feature_cols)
    return model.predict_proba(X)[:, 1][0]


def get_match_win_prob_cached(model, player_A, player_B, feature_cols, prob_cache):
    """
    Cached matchup probability.
    """
    
    A_id = player_A["player_id"]
    B_id = player_B["player_id"]
    
    key = (A_id, B_id)
    
    if key not in prob_cache:
        p_A = predict_player_A_win_prob(
            model,
            player_A,
            player_B,
            feature_cols
        )
        
        prob_cache[(A_id, B_id)] = p_A
        prob_cache[(B_id, A_id)] = 1 - p_A
    
    return prob_cache[key]

### Draw-based expected score optimizer

In [11]:
def solve_expected_score_draw_bracket(
    node,
    players,
    model,
    feature_cols,
    round_points,
    prob_cache=None
):
    """
    Solves the expected-score optimized bracket for a future draw.
    
    Returns:
    - probability each player wins this node
    - best bracket conditional on each possible winner
    """
    
    if prob_cache is None:
        prob_cache = {}
    
    current_round = node["round"]
    current_points = round_points[current_round]
    
    # Base case
    if current_round == "R128":
        p1 = players[node["p1_id"]]
        p2 = players[node["p2_id"]]
        
        p1_win_prob = get_match_win_prob_cached(
            model,
            p1,
            p2,
            feature_cols,
            prob_cache
        )
        
        node_win_probs = {
            p1["player_id"]: p1_win_prob,
            p2["player_id"]: 1 - p1_win_prob
        }
        
        best_by_winner = {}
        
        for p in [p1, p2]:
            pid = p["player_id"]
            win_prob = node_win_probs[pid]
            expected_points = current_points * win_prob
            
            pick = {
                "round": current_round,
                "round_index": node["round_index"],
                "player_1": p1["name"],
                "player_2": p2["name"],
                "pick_id": pid,
                "pick_name": p["name"],
                "win_prob": win_prob,
                "expected_points": expected_points
            }
            
            best_by_winner[pid] = {
                "player": p,
                "expected_score": expected_points,
                "picks": [pick]
            }
        
        return node_win_probs, best_by_winner
    
    # Recursive case
    left_probs, left_best = solve_expected_score_draw_bracket(
        node["left"],
        players,
        model,
        feature_cols,
        round_points,
        prob_cache
    )
    
    right_probs, right_best = solve_expected_score_draw_bracket(
        node["right"],
        players,
        model,
        feature_cols,
        round_points,
        prob_cache
    )
    
    node_win_probs = {}
    
    # Left-side players winning this section
    for left_id, p_left_reaches in left_probs.items():
        left_player = players[left_id]
        prob_wins_node_given_reaches = 0
        
        for right_id, p_right_reaches in right_probs.items():
            right_player = players[right_id]
            
            p_left_beats_right = get_match_win_prob_cached(
                model,
                left_player,
                right_player,
                feature_cols,
                prob_cache
            )
            
            prob_wins_node_given_reaches += (
                p_right_reaches * p_left_beats_right
            )
        
        node_win_probs[left_id] = (
            p_left_reaches * prob_wins_node_given_reaches
        )
    
    # Right-side players winning this section
    for right_id, p_right_reaches in right_probs.items():
        right_player = players[right_id]
        prob_wins_node_given_reaches = 0
        
        for left_id, p_left_reaches in left_probs.items():
            left_player = players[left_id]
            
            p_right_beats_left = get_match_win_prob_cached(
                model,
                right_player,
                left_player,
                feature_cols,
                prob_cache
            )
            
            prob_wins_node_given_reaches += (
                p_left_reaches * p_right_beats_left
            )
        
        node_win_probs[right_id] = (
            p_right_reaches * prob_wins_node_given_reaches
        )
    
    best_left_overall = max(
        left_best.values(),
        key=lambda x: x["expected_score"]
    )
    
    best_right_overall = max(
        right_best.values(),
        key=lambda x: x["expected_score"]
    )
    
    best_by_winner = {}
    
    # Conditional bracket if left-side player wins this section
    for left_id, left_option in left_best.items():
        picked_player = left_option["player"]
        opponent_pick = best_right_overall["player"]
        
        win_prob = node_win_probs[left_id]
        expected_points = current_points * win_prob
        
        current_pick = {
            "round": current_round,
            "round_index": node["round_index"],
            "player_1": picked_player["name"],
            "player_2": opponent_pick["name"],
            "pick_id": left_id,
            "pick_name": picked_player["name"],
            "win_prob": win_prob,
            "expected_points": expected_points
        }
        
        total_expected_score = (
            left_option["expected_score"]
            + best_right_overall["expected_score"]
            + expected_points
        )
        
        best_by_winner[left_id] = {
            "player": picked_player,
            "expected_score": total_expected_score,
            "picks": left_option["picks"] + best_right_overall["picks"] + [current_pick]
        }
    
    # Conditional bracket if right-side player wins this section
    for right_id, right_option in right_best.items():
        picked_player = right_option["player"]
        opponent_pick = best_left_overall["player"]
        
        win_prob = node_win_probs[right_id]
        expected_points = current_points * win_prob
        
        current_pick = {
            "round": current_round,
            "round_index": node["round_index"],
            "player_1": opponent_pick["name"],
            "player_2": picked_player["name"],
            "pick_id": right_id,
            "pick_name": picked_player["name"],
            "win_prob": win_prob,
            "expected_points": expected_points
        }
        
        total_expected_score = (
            best_left_overall["expected_score"]
            + right_option["expected_score"]
            + expected_points
        )
        
        best_by_winner[right_id] = {
            "player": picked_player,
            "expected_score": total_expected_score,
            "picks": best_left_overall["picks"] + right_option["picks"] + [current_pick]
        }
    
    return node_win_probs, best_by_winner

### Generate top-3 champion constrained brackets

In [12]:
def generate_top_n_constrained_brackets_for_draw(
    draw_df,
    match_df,
    neutral_df,
    tournament_name,
    year,
    tournament_start_date,
    feature_cols,
    round_points,
    top_n=3
):
    """
    Full production function:
    - trains model before tournament
    - builds current player features
    - loads draw
    - computes title probabilities
    - generates top-N champion-constrained optimized brackets
    """
    
    tournament_info = GRAND_SLAM_CONFIG[tournament_name]
    target_surface = tournament_info["surface"]
    draw_size = tournament_info["draw_size"]
    
    model = train_model_before_date(
        neutral_df,
        tournament_start_date,
        feature_cols
    )
    
    current_features = build_current_player_features_before_date(
        match_df,
        tournament_start_date,
        target_surface
    )
    
    draw_players, match_report = match_draw_to_player_features(
        draw_df,
        current_features,
        target_surface
    )
    
    root, nodes_by_round = build_draw_bracket_tree(
        draw_players,
        draw_size=draw_size
    )
    
    root_win_probs, best_by_champion = solve_expected_score_draw_bracket(
        root,
        draw_players,
        model,
        feature_cols,
        round_points
    )
    
    champ_probs_df = pd.DataFrame([
        {
            "player_id": pid,
            "player_name": draw_players[pid]["name"],
            "draw_slot": draw_players[pid]["draw_slot"],
            "rank": draw_players[pid]["rank"],
            "rank_points": draw_players[pid]["rank_points"],
            "surface_elo": draw_players[pid]["surface_elo"],
            "score_surface_elo": draw_players[pid]["score_surface_elo"],
            "championship_prob": prob
        }
        for pid, prob in root_win_probs.items()
    ])
    
    champ_probs_df = champ_probs_df.sort_values(
        "championship_prob",
        ascending=False
    ).reset_index(drop=True)
    
    champ_probs_df["champion_prob_rank"] = champ_probs_df.index + 1
    
    top_champs = champ_probs_df.head(top_n).copy()
    
    all_brackets = []
    bracket_summaries = []
    
    for _, champ_row in top_champs.iterrows():
        champ_id = champ_row["player_id"]
        champ_rank = int(champ_row["champion_prob_rank"])
        
        constrained_option = best_by_champion[champ_id]
        
        picks_df = pd.DataFrame(constrained_option["picks"])
        
        bracket_name = f"{tournament_name}_{year}_top_{champ_rank}_champion"
        
        picks_df["bracket_name"] = bracket_name
        picks_df["tournament"] = tournament_name
        picks_df["year"] = year
        picks_df["champion_prob_rank"] = champ_rank
        picks_df["constrained_champion_id"] = champ_id
        picks_df["constrained_champion_name"] = draw_players[champ_id]["name"]
        picks_df["constrained_champion_prob"] = champ_row["championship_prob"]
        picks_df["optimized_expected_score"] = constrained_option["expected_score"]
        picks_df["points_possible"] = picks_df["round"].map(round_points)
        
        all_brackets.append(picks_df)
        
        final_pick = picks_df[picks_df["round"] == "F"].iloc[0]
        
        bracket_summaries.append({
            "bracket_name": bracket_name,
            "tournament": tournament_name,
            "year": year,
            "champion_prob_rank": champ_rank,
            "champion": final_pick["pick_name"],
            "championship_prob": champ_row["championship_prob"],
            "optimized_expected_score": constrained_option["expected_score"]
        })
    
    brackets_df = pd.concat(all_brackets, ignore_index=True)
    bracket_summaries_df = pd.DataFrame(bracket_summaries)
    
    return {
        "brackets": brackets_df,
        "summaries": bracket_summaries_df,
        "championship_probabilities": champ_probs_df,
        "match_report": match_report,
        "nodes_by_round": nodes_by_round
    }

### Run it for any Grand Slam

In [22]:
TOURNAMENT_NAME = "Wimbledon"
YEAR = 2026
TOURNAMENT_START_DATE = "2026-06-29"

draw_df = pd.read_csv(f"{INPUT_DIR}/wimbledon_2026_draw.csv")

# Manual draw name / ID fixes

if "player_id" not in draw_df.columns:
    draw_df["player_id"] = pd.NA

draw_df.loc[
    draw_df["player_name"] == "Aleksandr Shevchenko",
    "player_name"
] = "Alexander Shevchenko"

draw_df.loc[
    draw_df["player_name"] == "Alexander Shevchenko",
    "player_id"
] = "S0H2"

results = generate_top_n_constrained_brackets_for_draw(
    draw_df=draw_df,
    match_df=match_df,
    neutral_df=neutral_df,
    tournament_name=TOURNAMENT_NAME,
    year=YEAR,
    tournament_start_date=TOURNAMENT_START_DATE,
    feature_cols=MODEL_FEATURE_COLS,
    round_points=ROUND_POINTS,
    top_n=3
)

brackets_2026 = results["brackets"]
summaries_2026 = results["summaries"]
champ_probs_2026 = results["championship_probabilities"]
match_report_2026 = results["match_report"]

Training cutoff: 2026-06-29
Training matches: 47136


In [23]:
summaries_2026

,bracket_name,tournament,year,champion_prob_rank,champion,championship_prob,optimized_expected_score
0,Wimbledon_2026_top_1_champion,Wimbledon,2026,1,Jannik Sinner,0.381258,2086.846980
1,Wimbledon_2026_top_2_champion,Wimbledon,2026,2,Novak Djokovic,0.167561,1874.186040
2,Wimbledon_2026_top_3_champion,Wimbledon,2026,3,Alexander Zverev,0.121693,1920.725206


In [24]:
champ_probs_2026.head(15)

,player_id,player_name,draw_slot,rank,rank_points,surface_elo,score_surface_elo,championship_prob,champion_prob_rank
0,S0AG,Jannik Sinner,1,1.0,14750.0,1789.493449,1797.486257,0.381258,1
1,D643,Novak Djokovic,64,4.0,4460.0,1973.031071,1980.540426,0.167561,2
2,Z355,Alexander Zverev,128,3.0,7190.0,1695.470889,1706.617374,0.121693,3
3,DH58,Alex de Minaur,65,6.0,4060.0,1700.786873,1714.256223,0.052969,4
4,FB98,Taylor Fritz,97,9.0,3635.0,1776.851360,1778.405310,0.029400,5
5,AG37,Felix Auger-Aliassime,33,4.0,4390.0,1597.306690,1606.443975,0.027222,6
6,MM58,Daniil Medvedev,32,7.0,3810.0,1701.172688,1706.484391,0.024717,7
7,S0S1,Ben Shelton,96,5.0,4070.0,1640.205157,1637.849684,0.021056,8
8,PL56,Tommy Paul,24,28.0,1645.0,1667.026200,1682.210695,0.019547,9
9,BK92,Alexander Bublik,112,11.0,3020.0,1675.624901,1675.504894,0.013080,10


In [25]:
match_report_2026["match_method"].value_counts()

match_method
name               124
unknown_default      3
player_id            1
Name: count, dtype: int64

In [26]:
match_report_2026[
    match_report_2026["match_method"] == "unknown_default"
]

,slot,draw_player_name,matched_player_id,matched_player_name,match_method,rank,rank_points,surface_elo,score_surface_elo
9,10,Felix Gill,UNKNOWN_SLOT_10,Felix Gill,unknown_default,NaN,NaN,1500.0,1500.0
13,14,Max Basing,UNKNOWN_SLOT_14,Max Basing,unknown_default,NaN,NaN,1500.0,1500.0
59,60,Martin Damm Jr,UNKNOWN_SLOT_60,Martin Damm Jr,unknown_default,NaN,NaN,1500.0,1500.0


### Export the 3 brackets

In [27]:
output_prefix = f"{TOURNAMENT_NAME.replace(' ', '_').lower()}_{YEAR}"

brackets_2026.to_csv(
    f"{output_prefix}_top3_constrained_brackets.csv",
    index=False
)

summaries_2026.to_csv(
    f"{output_prefix}_bracket_summaries.csv",
    index=False
)

champ_probs_2026.to_csv(
    f"{output_prefix}_championship_probabilities.csv",
    index=False
)

match_report_2026.to_csv(
    f"{output_prefix}_draw_match_report.csv",
    index=False
)

### Display by bracket and round

In [28]:
display_cols = [
    "bracket_name",
    "round",
    "round_index",
    "player_1",
    "player_2",
    "pick_name",
    "win_prob",
    "expected_points",
    "constrained_champion_name"
]

brackets_2026[display_cols].sort_values(
    ["bracket_name", "round", "round_index"]
)

,bracket_name,round,round_index,player_1,player_2,pick_name,win_prob,expected_points,constrained_champion_name
126,Wimbledon_2026_top_1_champion,F,0,Jannik Sinner,Alexander Zverev,Jannik Sinner,0.381258,244.005360,Jannik Sinner
30,Wimbledon_2026_top_1_champion,QF,0,Jannik Sinner,Tommy Paul,Jannik Sinner,0.630101,100.816173,Jannik Sinner
61,Wimbledon_2026_top_1_champion,QF,1,Felix Auger-Aliassime,Novak Djokovic,Novak Djokovic,0.470831,75.333038,Jannik Sinner
93,Wimbledon_2026_top_1_champion,QF,2,Alex de Minaur,Ben Shelton,Alex de Minaur,0.254492,40.718736,Jannik Sinner
124,Wimbledon_2026_top_1_champion,QF,3,Taylor Fritz,Alexander Zverev,Alexander Zverev,0.377759,60.441405,Jannik Sinner
...,...,...,...,...,...,...,...,...,...
368,Wimbledon_2026_top_3_champion,R64,29,Alex Michelsen,Francisco Cerundolo,Francisco Cerundolo,0.467298,9.345959,Alexander Zverev
372,Wimbledon_2026_top_3_champion,R64,30,Quentin Halys,Corentin Moutet,Corentin Moutet,0.314080,6.281605,Alexander Zverev
375,Wimbledon_2026_top_3_champion,R64,31,Valentin Royer,Alexander Zverev,Alexander Zverev,0.795143,15.902855,Alexander Zverev
316,Wimbledon_2026_top_3_champion,SF,0,Jannik Sinner,Novak Djokovic,Jannik Sinner,0.478364,153.076592,Alexander Zverev


In [30]:
# ============================================================
# PRETTY HTML BRACKET EXPORT
# Put this at the end of the bracket creator notebook
# ============================================================

from pathlib import Path
import pandas as pd
import html
import webbrowser

ROUND_ORDER = {
    "R128": 0,
    "R64": 1,
    "R32": 2,
    "R16": 3,
    "QF": 4,
    "SF": 5,
    "F": 6,
}

QUARTERS = [
    ("Quarter 1", 0, 15, 0),
    ("Quarter 2", 16, 31, 1),
    ("Quarter 3", 32, 47, 2),
    ("Quarter 4", 48, 63, 3),
]


def esc(x):
    if pd.isna(x):
        return ""
    return html.escape(str(x))


def fmt_prob(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.1%}"


def get_row(bracket_df, round_name, round_index):
    rows = bracket_df[
        (bracket_df["round"] == round_name)
        & (bracket_df["round_index"] == round_index)
    ]
    if rows.empty:
        return None
    return rows.iloc[0]


def player_line(player_name, pick_name):
    css = "picked" if str(player_name) == str(pick_name) else "not-picked"
    return f'<div class="{css}">{esc(player_name)}</div>'


def matchup_cell(row):
    if row is None:
        return '<div class="match-card missing">Missing</div>'

    p1 = row["player_1"]
    p2 = row["player_2"]
    pick = row["pick_name"]
    prob = fmt_prob(row["win_prob"])

    return f"""
    <div class="match-card">
        {player_line(p1, pick)}
        {player_line(p2, pick)}
        <div class="pick-meta">Pick: <b>{esc(pick)}</b> · {prob}</div>
    </div>
    """


def winner_cell(row, label=None):
    if row is None:
        return '<div class="winner-card missing">Missing</div>'

    pick = row["pick_name"]
    prob = fmt_prob(row["win_prob"])
    label_html = f'<div class="round-label">{esc(label)}</div>' if label else ""

    return f"""
    <div class="winner-card">
        {label_html}
        <div class="winner-name">{esc(pick)}</div>
        <div class="pick-meta">{prob}</div>
    </div>
    """


def make_quarter_table(bracket_df, quarter_name, r128_start, r128_end, qf_index):
    rows_html = []

    for r128_index in range(r128_start, r128_end + 1):
        local_i = r128_index - r128_start

        r128 = get_row(bracket_df, "R128", r128_index)
        r64 = get_row(bracket_df, "R64", r128_index // 2)
        r32 = get_row(bracket_df, "R32", r128_index // 4)
        r16 = get_row(bracket_df, "R16", r128_index // 8)
        qf = get_row(bracket_df, "QF", qf_index)

        row = "<tr>"
        row += f"<td>{matchup_cell(r128)}</td>"

        if local_i % 2 == 0:
            row += f'<td rowspan="2">{winner_cell(r64, "R64")}</td>'

        if local_i % 4 == 0:
            row += f'<td rowspan="4">{winner_cell(r32, "R32")}</td>'

        if local_i % 8 == 0:
            row += f'<td rowspan="8">{winner_cell(r16, "R16")}</td>'

        if local_i % 16 == 0:
            row += f'<td rowspan="16">{winner_cell(qf, "QF")}</td>'

        row += "</tr>"
        rows_html.append(row)

    return f"""
    <section class="quarter-section">
        <h3>{esc(quarter_name)}</h3>
        <table class="bracket-table">
            <thead>
                <tr>
                    <th>Round 1</th>
                    <th>Round 2</th>
                    <th>Round 3</th>
                    <th>Round 4</th>
                    <th>Quarterfinalist</th>
                </tr>
            </thead>
            <tbody>
                {''.join(rows_html)}
            </tbody>
        </table>
    </section>
    """


def make_late_rounds(bracket_df):
    sf0 = get_row(bracket_df, "SF", 0)
    sf1 = get_row(bracket_df, "SF", 1)
    final = get_row(bracket_df, "F", 0)

    return f"""
    <section class="late-rounds">
        <h3>Final Four Path</h3>
        <div class="late-grid">
            <div>
                <div class="mini-heading">Semifinal 1</div>
                {matchup_cell(sf0)}
            </div>
            <div>
                <div class="mini-heading">Semifinal 2</div>
                {matchup_cell(sf1)}
            </div>
            <div>
                <div class="mini-heading">Final</div>
                {matchup_cell(final)}
            </div>
            <div>
                <div class="mini-heading">Champion</div>
                {winner_cell(final)}
            </div>
        </div>
    </section>
    """


def make_bracket_section(bracket_df, bracket_name):
    bracket_df = bracket_df.copy()
    bracket_df["round_order"] = bracket_df["round"].map(ROUND_ORDER)
    bracket_df = bracket_df.sort_values(["round_order", "round_index"])

    champion = bracket_df.loc[bracket_df["round"] == "F", "pick_name"].iloc[0]
    constrained_champion = bracket_df["constrained_champion_name"].iloc[0]
    constrained_prob = bracket_df["constrained_champion_prob"].iloc[0]
    expected_score = bracket_df["optimized_expected_score"].iloc[0]
    champ_rank = int(bracket_df["champion_prob_rank"].iloc[0])

    quarters_html = "\n".join(
        make_quarter_table(bracket_df, quarter_name, r128_start, r128_end, qf_index)
        for quarter_name, r128_start, r128_end, qf_index in QUARTERS
    )

    return f"""
    <article class="bracket-section" id="{esc(bracket_name)}">
        <div class="bracket-header">
            <div>
                <h2>{esc(bracket_name)}</h2>
                <p>Champion constraint rank #{champ_rank}: <b>{esc(constrained_champion)}</b> ({fmt_prob(constrained_prob)})</p>
            </div>
            <div class="summary-cards">
                <div class="summary-card">
                    <span>Champion</span>
                    <b>{esc(champion)}</b>
                </div>
                <div class="summary-card">
                    <span>Expected score</span>
                    <b>{float(expected_score):.1f}</b>
                </div>
            </div>
        </div>

        {make_late_rounds(bracket_df)}

        {quarters_html}
    </article>
    """


def make_pretty_bracket_html(df):
    required_cols = {
        "round",
        "round_index",
        "player_1",
        "player_2",
        "pick_name",
        "win_prob",
        "bracket_name",
        "champion_prob_rank",
        "constrained_champion_name",
        "constrained_champion_prob",
        "optimized_expected_score",
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    bracket_names = list(df["bracket_name"].drop_duplicates())

    nav_links = "\n".join(
        f'<a href="#{esc(name)}">{esc(name)}</a>' for name in bracket_names
    )

    sections = "\n".join(
        make_bracket_section(df[df["bracket_name"] == name], name)
        for name in bracket_names
    )

    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Wimbledon 2026 Optimized Brackets</title>
<style>
    :root {{
        --green: #0B3D2E;
        --light-green: #EAF4EA;
        --gold: #D6B25E;
        --gray: #667085;
        --border: #D9E2D6;
        --bg: #F7F9F5;
        --card: #FFFFFF;
    }}

    body {{
        margin: 0;
        font-family: Arial, Helvetica, sans-serif;
        background: var(--bg);
        color: #172B22;
    }}

    header {{
        background: linear-gradient(135deg, var(--green), #176B4D);
        color: white;
        padding: 28px 36px;
        position: sticky;
        top: 0;
        z-index: 10;
        box-shadow: 0 4px 18px rgba(0,0,0,0.16);
    }}

    header h1 {{
        margin: 0 0 8px 0;
        font-size: 30px;
    }}

    header p {{
        margin: 0;
        color: rgba(255,255,255,0.86);
    }}

    nav {{
        margin-top: 16px;
        display: flex;
        flex-wrap: wrap;
        gap: 10px;
    }}

    nav a {{
        color: white;
        text-decoration: none;
        border: 1px solid rgba(255,255,255,0.4);
        border-radius: 999px;
        padding: 7px 12px;
        font-size: 13px;
    }}

    main {{
        padding: 28px 36px 56px 36px;
    }}

    .bracket-section {{
        background: var(--card);
        border: 1px solid var(--border);
        border-radius: 18px;
        padding: 22px;
        margin-bottom: 36px;
        box-shadow: 0 6px 20px rgba(0,0,0,0.06);
    }}

    .bracket-header {{
        display: flex;
        justify-content: space-between;
        align-items: stretch;
        gap: 18px;
        border-bottom: 1px solid var(--border);
        padding-bottom: 16px;
        margin-bottom: 18px;
    }}

    .bracket-header h2 {{
        margin: 0 0 8px 0;
        font-size: 24px;
        color: var(--green);
    }}

    .bracket-header p {{
        margin: 0;
        color: var(--gray);
    }}

    .summary-cards {{
        display: flex;
        gap: 12px;
        min-width: 330px;
    }}

    .summary-card {{
        flex: 1;
        border: 1px solid var(--border);
        background: var(--light-green);
        border-radius: 14px;
        padding: 12px;
    }}

    .summary-card span {{
        display: block;
        font-size: 12px;
        color: var(--gray);
        margin-bottom: 6px;
        text-transform: uppercase;
        letter-spacing: 0.04em;
    }}

    .summary-card b {{
        display: block;
        font-size: 18px;
        color: var(--green);
    }}

    .late-rounds {{
        margin-bottom: 24px;
    }}

    .late-grid {{
        display: grid;
        grid-template-columns: repeat(4, minmax(180px, 1fr));
        gap: 14px;
    }}

    .mini-heading {{
        font-size: 12px;
        font-weight: 700;
        color: var(--gray);
        text-transform: uppercase;
        letter-spacing: 0.04em;
        margin-bottom: 6px;
    }}

    .quarter-section {{
        margin-top: 26px;
    }}

    .quarter-section h3, .late-rounds h3 {{
        color: var(--green);
        font-size: 18px;
        margin: 0 0 10px 0;
    }}

    .bracket-table {{
        width: 100%;
        border-collapse: separate;
        border-spacing: 8px;
        table-layout: fixed;
    }}

    .bracket-table th {{
        text-align: left;
        font-size: 12px;
        color: var(--gray);
        text-transform: uppercase;
        letter-spacing: 0.04em;
        padding: 0 6px 4px 6px;
    }}

    .bracket-table td {{
        vertical-align: middle;
    }}

    .match-card, .winner-card {{
        border: 1px solid var(--border);
        border-radius: 12px;
        background: white;
        padding: 9px 10px;
        min-height: 46px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.035);
    }}

    .winner-card {{
        background: #FAFCFA;
        display: flex;
        flex-direction: column;
        justify-content: center;
        height: calc(100% - 20px);
    }}

    .picked {{
        font-weight: 800;
        color: var(--green);
        font-size: 14px;
        line-height: 1.25;
    }}

    .not-picked {{
        color: #7A8699;
        font-size: 13px;
        line-height: 1.25;
    }}

    .pick-meta {{
        margin-top: 6px;
        color: var(--gray);
        font-size: 11px;
    }}

    .round-label {{
        color: var(--gold);
        font-size: 11px;
        font-weight: 800;
        text-transform: uppercase;
        letter-spacing: 0.04em;
        margin-bottom: 4px;
    }}

    .winner-name {{
        color: var(--green);
        font-size: 15px;
        font-weight: 800;
    }}

    .missing {{
        color: #B42318;
        background: #FFF4F2;
    }}

    @media print {{
        header {{
            position: static;
        }}

        .bracket-section {{
            page-break-after: always;
            box-shadow: none;
        }}

        body {{
            background: white;
        }}
    }}
</style>
</head>
<body>
<header>
    <h1>Wimbledon 2026 Optimized Brackets</h1>
    <p>Top-3 champion-constrained expected-score brackets. Bold green names are the picks to fill in.</p>
    <nav>
        {nav_links}
    </nav>
</header>
<main>
    {sections}
</main>
</body>
</html>
"""


# ============================================================
# CREATE HTML FILE
# ============================================================

# Use the in-memory dataframe from your bracket producer.
# If yours is named something else, replace brackets_2026 below.
pretty_html = make_pretty_bracket_html(brackets_2026)

output_path = Path("wimbledon_2026_pretty_brackets.html")
output_path.write_text(pretty_html, encoding="utf-8")

print(f"Pretty bracket saved to: {output_path.resolve()}")

# Open automatically in your default browser
webbrowser.open(output_path.resolve().as_uri())

Pretty bracket saved to: C:\Users\ishan\Downloads\Wimbledon Prediction Model\wimbledon_2026_pretty_brackets.html


True